Object-oriented programming is a way of organising code around the things your program works with rather than the steps it takes. You define classes — blueprints that bundle together data (attributes) and behaviour (methods) — and create instances of those classes to represent concrete objects. The result is code that models the problem domain more directly, is easier to extend without breaking existing parts, and keeps related logic together rather than scattered. This notebook covers everything from defining a basic class through inheritance, properties, dunder methods, and dataclasses.

## Classes and Instances

A **class** is a blueprint — it defines what data an object holds and what it can do. An **instance** is a concrete object built from that blueprint. Every value in Python is already an instance of some class: `42` is an `int`, `"hello"` is a `str`, `[1, 2]` is a `list`.

Define a class with the `class` keyword. The `__init__` method is the **initialiser** — it runs automatically when a new instance is created and sets up the instance's initial state. `self` is the first parameter of every instance method; Python passes the instance itself as that argument automatically.

In [ ]:
class BankAccount:
    def __init__(self, owner, balance=0.0):
        self.owner   = owner     # instance attribute
        self.balance = balance

    def deposit(self, amount):
        self.balance += amount
        print(f"Deposited {amount:.2f} — new balance: {self.balance:.2f}")

    def withdraw(self, amount):
        if amount > self.balance:
            raise ValueError(f"Insufficient funds: have {self.balance:.2f}")
        self.balance -= amount
        print(f"Withdrew {amount:.2f} — new balance: {self.balance:.2f}")

    def __str__(self):
        return f"BankAccount({self.owner!r}, balance={self.balance:.2f})"


# Create instances
alice = BankAccount("Alice", 1000.0)
bob   = BankAccount("Bob")

alice.deposit(250.0)
alice.withdraw(100.0)
print(alice)           # uses __str__
print(alice.owner, alice.balance)

# Each instance has its own independent state
print(bob.balance)     # 0.0

## Class Attributes vs Instance Attributes

**Instance attributes** are set on `self` — each instance gets its own copy. **Class attributes** are defined directly in the class body — they are shared across all instances.

Reading a name on an instance first checks the instance's own `__dict__`, then the class. Writing always creates or updates the instance's attribute, leaving the class attribute untouched.

In [ ]:
class Connection:
    MAX_RETRIES  = 3          # class attribute — shared by all instances
    _instance_count = 0       # class attribute used as a counter

    def __init__(self, host, port):
        self.host = host      # instance attribute
        self.port = port
        Connection._instance_count += 1

    @classmethod
    def count(cls):
        return cls._instance_count

c1 = Connection("db.example.com", 5432)
c2 = Connection("cache.example.com", 6379)

print(Connection.MAX_RETRIES)   # 3 — accessed on the class
print(c1.MAX_RETRIES)           # 3 — also accessible on the instance
print(Connection.count())       # 2

# Assigning to c1.MAX_RETRIES creates an instance attribute — doesn't touch the class
c1.MAX_RETRIES = 5
print(c1.MAX_RETRIES)           # 5 — instance's own copy
print(c2.MAX_RETRIES)           # 3 — class value unchanged

## `@classmethod` and `@staticmethod`

- **`@classmethod`**: receives the class as its first argument (`cls`). Used for alternative constructors and factory methods that produce instances.
- **`@staticmethod`**: receives no implicit first argument. A plain function namespaced inside the class because it is logically related, but doesn't need access to the instance or the class.

In [ ]:
class Temperature:
    def __init__(self, celsius):
        self.celsius = celsius

    # Alternative constructors — classmethods are the idiomatic factory pattern
    @classmethod
    def from_fahrenheit(cls, f):
        return cls((f - 32) * 5 / 9)

    @classmethod
    def from_kelvin(cls, k):
        return cls(k - 273.15)

    # Utility that belongs here logically but needs no instance or class
    @staticmethod
    def is_valid_celsius(value):
        return -273.15 <= value

    def __str__(self):
        return f"{self.celsius:.2f}°C"


boiling  = Temperature(100)
body     = Temperature.from_fahrenheit(98.6)
absolute = Temperature.from_kelvin(0)

print(boiling)    # 100.00°C
print(body)       # 37.00°C
print(absolute)   # -273.15°C

print(Temperature.is_valid_celsius(-300))  # False
print(Temperature.is_valid_celsius(20))    # True

## Properties

A **property** lets you expose an attribute-like interface while running custom getter, setter, and deleter logic behind the scenes. The caller accesses it like an attribute (`obj.name`) but you control exactly what happens.

This is how Python implements encapsulation cleanly: start with a plain attribute; if you ever need validation or computed values, convert it to a property without changing the public API.

In [ ]:
class Circle:
    def __init__(self, radius):
        self.radius = radius   # setter is called here too

    @property
    def radius(self):
        return self._radius

    @radius.setter
    def radius(self, value):
        if value < 0:
            raise ValueError(f"Radius cannot be negative: {value}")
        self._radius = value

    @property
    def area(self):
        import math
        return math.pi * self._radius ** 2

    @property
    def diameter(self):
        return self._radius * 2

    @diameter.setter
    def diameter(self, value):
        self.radius = value / 2   # routes through radius setter for validation


c = Circle(5)
print(c.radius)     # 5
print(c.diameter)   # 10
print(f"{c.area:.4f}")  # 78.5398

c.diameter = 20
print(c.radius)     # 10.0

try:
    c.radius = -1
except ValueError as e:
    print(e)

## Dunder (Magic) Methods

**Dunder methods** (double-underscore on both sides) let your objects respond to built-in Python operations — `len()`, `str()`, `+`, `==`, `<`, `in`, and more. They make your classes feel like first-class Python citizens.

| Method | Triggered by |
|--------|--------------|
| `__init__` | `MyClass(...)` |
| `__str__` | `str(obj)`, `print(obj)` |
| `__repr__` | `repr(obj)`, interactive prompt |
| `__len__` | `len(obj)` |
| `__eq__` | `obj == other` |
| `__lt__` | `obj < other` |
| `__add__` | `obj + other` |
| `__contains__` | `item in obj` |
| `__getitem__` | `obj[key]` |
| `__iter__` | `for item in obj` |
| `__bool__` | `bool(obj)`, `if obj:` |

In [ ]:
class Vector:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __repr__(self):
        # repr should be unambiguous — ideally eval-able
        return f"Vector({self.x}, {self.y})"

    def __str__(self):
        # str can be friendlier / more readable
        return f"({self.x}, {self.y})"

    def __add__(self, other):
        return Vector(self.x + other.x, self.y + other.y)

    def __sub__(self, other):
        return Vector(self.x - other.x, self.y - other.y)

    def __mul__(self, scalar):
        return Vector(self.x * scalar, self.y * scalar)

    def __eq__(self, other):
        return isinstance(other, Vector) and self.x == other.x and self.y == other.y

    def __abs__(self):
        import math
        return math.sqrt(self.x ** 2 + self.y ** 2)

    def __bool__(self):
        return self.x != 0 or self.y != 0


v1 = Vector(2, 3)
v2 = Vector(1, -1)

print(repr(v1))         # Vector(2, 3)
print(str(v1))          # (2, 3)
print(v1 + v2)          # (3, 2)
print(v1 - v2)          # (1, 4)
print(v1 * 3)           # (6, 9)
print(v1 == Vector(2, 3))  # True
print(abs(v1))          # 3.605...
print(bool(Vector(0, 0)))  # False — zero vector is falsy

In [ ]:
class Playlist:
    def __init__(self, name):
        self.name  = name
        self._songs = []

    def add(self, song):
        self._songs.append(song)

    def __len__(self):
        return len(self._songs)

    def __contains__(self, song):
        return song in self._songs

    def __getitem__(self, index):
        return self._songs[index]

    def __iter__(self):
        return iter(self._songs)

    def __repr__(self):
        return f"Playlist({self.name!r}, {len(self)} songs)"


pl = Playlist("Road Trip")
pl.add("Bohemian Rhapsody")
pl.add("Hotel California")
pl.add("Stairway to Heaven")

print(len(pl))                         # 3
print("Hotel California" in pl)        # True
print(pl[0])                           # Bohemian Rhapsody
print(pl[-1])                          # Stairway to Heaven

for song in pl:                        # __iter__ makes it work in for loops
    print(f"  ♪ {song}")

## Encapsulation — `_` and `__`

Python has no private keyword — everything is technically accessible. Instead, the community uses naming conventions to signal intent:

- **`_name`** — single underscore: *by convention* private. "I intended this for internal use; access it if you must, but don't rely on it."
- **`__name`** — double underscore: **name mangling**. Python rewrites `__name` defined in class `Foo` to `_Foo__name`. This prevents accidental overriding in subclasses — not true privacy, but effective collision avoidance.

In [ ]:
class SafeCounter:
    def __init__(self):
        self._value   = 0     # conventional private — don't touch from outside
        self.__secret = 42    # mangled — becomes _SafeCounter__secret

    def increment(self):
        self._value += 1

    def value(self):
        return self._value


c = SafeCounter()
c.increment()
c.increment()
print(c.value())     # 2

# Single underscore — accessible but frowned upon
print(c._value)      # 2 — works, but violates the convention

# Double underscore — name is mangled
# print(c.__secret)  # AttributeError: 'SafeCounter' object has no attribute '__secret'
print(c._SafeCounter__secret)   # 42 — still accessible if you know the mangled name

## Inheritance

Inheritance lets one class (**subclass** / **child**) build on another (**superclass** / **parent**). The subclass inherits all the parent's methods and attributes and can:
- Use them as-is.
- **Override** them — replace with a different implementation.
- **Extend** them — call the parent version with `super()` and add extra behaviour.

Use inheritance to model **is-a** relationships: a `Dog` *is a* `Animal`. For **has-a** relationships (a `Car` *has a* `Engine`), use composition instead.

In [ ]:
class Animal:
    def __init__(self, name, species):
        self.name    = name
        self.species = species

    def speak(self):
        return "..."

    def describe(self):
        return f"{self.name} is a {self.species} and says '{self.speak()}'"

    def __repr__(self):
        return f"{type(self).__name__}({self.name!r})"


class Dog(Animal):
    def __init__(self, name, breed):
        super().__init__(name, species="dog")   # call parent __init__
        self.breed = breed                      # add Dog-specific attribute

    def speak(self):
        return "Woof!"

    def fetch(self, item):
        return f"{self.name} fetches the {item}"


class Cat(Animal):
    def __init__(self, name, indoor=True):
        super().__init__(name, species="cat")
        self.indoor = indoor

    def speak(self):
        return "Meow!"


rex   = Dog("Rex", "German Shepherd")
kitty = Cat("Kitty")

print(rex.describe())    # Rex is a dog and says 'Woof!'
print(kitty.describe())  # Kitty is a cat and says 'Meow!'
print(rex.fetch("ball"))
print(repr(rex))

In [ ]:
# isinstance() and issubclass() — check the class hierarchy
print(isinstance(rex, Dog))     # True
print(isinstance(rex, Animal))  # True — Dog is an Animal
print(isinstance(rex, Cat))     # False

print(issubclass(Dog, Animal))  # True
print(issubclass(Cat, Dog))     # False

# type() gives the exact class, not the hierarchy
print(type(rex) is Dog)         # True
print(type(rex) is Animal)      # False

In [ ]:
# Extending a parent method with super()
class SavingsAccount(BankAccount):
    INTEREST_RATE = 0.03

    def __init__(self, owner, balance=0.0):
        super().__init__(owner, balance)   # delegate to BankAccount.__init__
        self.interest_earned = 0.0

    def apply_interest(self):
        interest = self.balance * self.INTEREST_RATE
        self.deposit(interest)             # inherits deposit() from BankAccount
        self.interest_earned += interest

    def __str__(self):
        base = super().__str__()           # reuse parent __str__
        return f"{base}, interest_earned={self.interest_earned:.2f}"


savings = SavingsAccount("Alice", 1000.0)
savings.apply_interest()
print(savings)

## Multiple Inheritance and MRO

Python allows a class to inherit from more than one parent. When the same method name exists in multiple parents, Python uses the **Method Resolution Order (MRO)** — a deterministic left-to-right, depth-first search that ensures each class in the hierarchy appears only once. The MRO is accessible via `ClassName.__mro__`.

Multiple inheritance is powerful but should be used sparingly. The most common legitimate use is **mixins** — small classes that add a specific capability without representing a meaningful standalone concept.

In [ ]:
class JSONMixin:
    """Mixin: adds .to_json() to any class with a __dict__."""
    def to_json(self):
        import json
        return json.dumps(self.__dict__, default=str)

class TimestampMixin:
    """Mixin: records creation time."""
    def __init__(self, *args, **kwargs):
        import datetime
        super().__init__(*args, **kwargs)
        self.created_at = datetime.datetime.now().isoformat()


class User(TimestampMixin, JSONMixin):
    def __init__(self, name, email):
        super().__init__()   # cooperative — calls TimestampMixin.__init__
        self.name  = name
        self.email = email


user = User("Alice", "alice@example.com")
print(user.name)
print(user.created_at)
print(user.to_json())

# Inspect the MRO
print([c.__name__ for c in User.__mro__])

## Dataclasses

The `@dataclass` decorator (Python 3.7+) auto-generates `__init__`, `__repr__`, and `__eq__` from a class's annotated fields — eliminating the boilerplate for classes that are primarily data containers.

Options on `@dataclass`:
- `order=True` — generates `__lt__`, `__le__`, `__gt__`, `__ge__` for sorting.
- `frozen=True` — makes instances immutable (like a named tuple with type hints).
- `slots=True` (Python 3.10+) — uses `__slots__` for lower memory footprint.

In [ ]:
from dataclasses import dataclass, field

@dataclass
class Point:
    x: float
    y: float

    def distance_to(self, other: "Point") -> float:
        import math
        return math.sqrt((self.x - other.x) ** 2 + (self.y - other.y) ** 2)


p1 = Point(1.0, 2.0)
p2 = Point(4.0, 6.0)

print(p1)                   # Point(x=1.0, y=2.0)  — __repr__ auto-generated
print(p1 == Point(1.0, 2.0))  # True              — __eq__ auto-generated
print(p1.distance_to(p2))   # 5.0

In [ ]:
@dataclass(order=True)
class Product:
    price: float
    name:  str
    tags:  list = field(default_factory=list)   # mutable default — use field()

    def apply_discount(self, pct):
        self.price = round(self.price * (1 - pct / 100), 2)


p1 = Product(9.99,  "Widget",  ["sale"])
p2 = Product(24.99, "Gadget")
p3 = Product(4.50,  "Donut")

print(sorted([p1, p2, p3]))   # sorted by price (first field) — order=True

p1.apply_discount(10)
print(p1)   # price updated

In [ ]:
# frozen=True — immutable dataclass, hashable
@dataclass(frozen=True)
class Coordinate:
    lat:  float
    lon:  float
    name: str = ""

tokyo  = Coordinate(35.6762, 139.6503, "Tokyo")
london = Coordinate(51.5074, -0.1278,  "London")

print(tokyo)

# Frozen dataclasses are hashable — can be used as dict keys or in sets
distances = {tokyo: 0, london: 9559}
print(distances[tokyo])

# Attempt to mutate raises FrozenInstanceError
try:
    tokyo.lat = 0
except Exception as e:
    print(f"{type(e).__name__}: {e}")

## Summary

- A **class** is a blueprint; an **instance** is a concrete object. `__init__` sets up initial state. `self` is the instance itself, passed automatically.
- **Instance attributes** live on `self` — unique per object. **Class attributes** live on the class — shared by all instances.
- **`@classmethod`** receives the class (`cls`) — use for alternative constructors. **`@staticmethod`** receives nothing — a plain helper namespaced in the class.
- **`@property`** exposes attribute-like access with getter/setter logic. Start with plain attributes; upgrade to properties when validation or computation is needed — the public API stays the same.
- **Dunder methods** make objects respond to built-in operations. Implement `__repr__` for debugging (unambiguous), `__str__` for display (readable), and others like `__eq__`, `__lt__`, `__add__`, `__len__`, `__iter__` to integrate with Python's operators and functions.
- **Encapsulation**: `_name` is conventional private (access discouraged). `__name` triggers name mangling to `_ClassName__name` — collision avoidance in inheritance hierarchies.
- **Inheritance** models *is-a* relationships. Use `super()` to call the parent's implementation. Override methods to change behaviour; extend them to add to it. Use `isinstance()` to check the hierarchy, not `type() is`.
- **Multiple inheritance**: Python resolves method lookup via MRO (left-to-right, C3 linearisation). Use mixins for composable capabilities.
- **`@dataclass`** auto-generates `__init__`, `__repr__`, and `__eq__` from annotated fields. Use `field(default_factory=...)` for mutable defaults. `order=True` adds comparison methods. `frozen=True` makes instances immutable and hashable.